In [0]:
storage_account_name = "bankingprojectstacc"
storage_account_key = dbutils.secrets.get(scope = 'keyvault-scope', key = 'storage-access-key')

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
df = spark.read.format("json").option("InferSchema", "True").load(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/raw_banking/")
display(df)

In [0]:
from pyspark.sql.functions import col, explode_outer

df_exploded = df.select(
    col("info"),
    explode_outer(col("results")).alias("result")
)

display(df_exploded)

In [0]:
df_final = df_exploded.select(
    col("info.*"),
    col("result.*")
)

display(df_final)

In [0]:
df_final = (
    df_final
    # Id
    .withColumn("id_name", col("id.name"))
    .withColumn("id_value", col("id.value"))
    .drop("id")
    # Location
    .withColumn("city", col("location.city"))
    .withColumn("coordinates", col("location.coordinates"))
    .withColumn("coordinate_latitude", col("location.coordinates.latitude"))
    .withColumn("coordinate_longitude", col("location.coordinates.longitude"))
    .withColumn("country", col("location.country"))
    .withColumn("postcode", col("location.postcode"))
    .withColumn("state", col("location.state"))
    .withColumn("street_number", col("location.street.number"))
    .withColumn("street_name", col("location.street.name"))
    .withColumn("street", col("location.street"))
    .withColumn("timezone_description", col("location.timezone.description"))
    .withColumn("timezone_offset", col("location.timezone.offset"))
    .withColumn("timezone", col("location.timezone"))
    .drop("location", "coordinates", "street", "timezone")
    # Login
    .withColumn("login_md5", col("login.md5"))
    .withColumn("login_password", col("login.password"))
    .withColumn("login_salt", col("login.salt"))
    .withColumn("login_sha1", col("login.sha1"))
    .withColumn("login_sha256", col("login.sha256"))
    .withColumn("login_username", col("login.username"))
    .withColumn("login_uuid", col("login.uuid"))
    .drop("login")
    # Name
    .withColumn("name_first", col("name.first"))
    .withColumn("name_last", col("name.last"))
    .withColumn("name_title", col("name.title"))
    .drop("name")
    # Picture
    .withColumn("picture_large", col("picture.large"))
    .withColumn("picture_medium", col("picture.medium"))
    .withColumn("picture_thumbnail", col("picture.thumbnail"))
    .drop("picture")
    # Registered
    .withColumn("registered_age", col("registered.age"))
    .withColumn("registered_date", col("registered.date"))
    .drop("registered")
    #dob
    .withColumn("dob_age", col("dob.age"))
    .withColumn("dob_date", col("dob.date"))
    .drop("dob")
)

display(df_final)

In [0]:
import random
from pyspark.sql.functions import col, expr
from datetime import datetime, timedelta

user_ids = [row["login_uuid"] for row in df_final.select("login_uuid").collect()]

transactions = []
transaction_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment']

for i in range(1, 5000):  # 5000 fake transactions
    u_id = random.choice(user_ids)
    t_type = random.choice(transaction_types)
    amount = round(random.uniform(10.0, 5000.0), 2)
    days_offset = random.randint(0, 30)
    t_date = (datetime.now() - timedelta(days=days_offset)).strftime("%Y-%m-%d %H:%M:%S")
    
    transactions.append((f"TXN_{1000+i}", u_id, t_type, amount, t_date))

df_transactions = spark.createDataFrame(
    transactions, 
    ["transaction_id", "user_id", "transaction_type", "amount", "transaction_timestamp"]
)

df_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_bankingproject.default.transactions")

print("Staging Transactions Table successfully created!")

In [0]:
display(df_transactions)

In [0]:
df_final.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("db_bankingproject.default.raw_banking")